# Pipeline de Preprocesamiento
## Prediccion de Default en Prestamos - LendingClub (2007-2018)

---

**Asignatura:** Programacion para la Ciencia de Datos  
**Evaluacion:** Proyecto Integrador - Final Transversal  
**Fecha:** Julio 2026  

**Equipo:** [Nombres de los integrantes]

---

### Objetivo del Pipeline

Este notebook implementa el pipeline ETL completo para transformar los datos
crudos de LendingClub en datasets limpios y listos para modelado. Las etapas
cubren:

1. **Carga exploratoria** de datos crudos y muestras
2. **Limpieza** de nombres de columnas y estandarizacion de formatos
3. **Construccion de variable objetivo** (`bad_loan`) a partir de `loan_status`
4. **Parseo** de fechas, montos monetarios, porcentajes y plazos
5. **Feature engineering** (FICO score, hardship, debt settlement)
6. **Eliminacion de columnas con leakage** (datos posteriores al prestamo)
7. **Manejo de valores faltantes** y tipos de datos
8. **Integracion** de datos aceptados y rechazados
9. **Exportacion** a `data/processed/` para consumo en modelado

### Pipeline ETL

```
raw/                     interim/                 processed/
accepted_2007_to_2018Q4  accepted_sample_5k  -->  accepted_clean
rejected_2007_to_2018Q4  rejected_sample_5k  -->  rejected_clean
                                                  combined_summary
```

---

In [84]:
import pandas as pd
import numpy as np
import os
import logging
import warnings
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings("ignore")

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

print("Librerias cargadas exitosamente")

Librerias cargadas exitosamente


In [85]:
RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"
INTERIM_DIR = "../data/interim"


os.makedirs(PROCESSED_DIR, exist_ok=True)
print(f"Directorios configurados. Output: {PROCESSED_DIR}/")

Directorios configurados. Output: ../data/processed/


---
## 1. Carga de Datos

Cargamos los datasets de prestamos ACEPTADOS (con outcome conocido) y
RECHAZADOS (solicitudes denegadas). Por defecto usamos muestras de 5,000
registros para iteracion rapida. Para produccion, cambiar `sample=False`
para procesar el dataset completo (~2.26M registros aceptados, ~27.7M rechazados).

### Estructura esperada

| Dataset | Origen | Filas | Columnas |
|---------|--------|-------|----------|
| Aceptados | `interim/accepted_sample_5k.csv` | 5,000 | 151 |
| Rechazados | `interim/rejected_sample_5k.csv` | 5,000 | 9 |

In [86]:
sample = True  # True para cargar solo 5K muestras

path_acc = os.path.join(INTERIM_DIR, "accepted_sample_20k.csv") if sample else os.path.join(RAW_DIR, "accepted_2007_to_2018Q4.csv")
path_rej = os.path.join(INTERIM_DIR, "rejected_sample_20k.csv") if sample else os.path.join(RAW_DIR, "rejected_2007_to_2018Q4.csv")

df_acc = pd.read_csv(path_acc, low_memory=False)
df_rej = pd.read_csv(path_rej, low_memory=False)

print(f"Dataset ACEPTADOS: {df_acc.shape[0]:,} filas x {df_acc.shape[1]} columnas")
print(f"Dataset RECHAZADOS: {df_rej.shape[0]:,} filas x {df_rej.shape[1]} columnas")

Dataset ACEPTADOS: 20,000 filas x 151 columnas
Dataset RECHAZADOS: 20,000 filas x 9 columnas


In [87]:
print("=" * 70)
print("VISTA PREVIA - DATOS ACEPTADOS")
print("=" * 70)
display(df_acc.head(3))
print(f"\nColumnas: {list(df_acc.columns)}")

VISTA PREVIA - DATOS ACEPTADOS


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,67849662,NaN,4225.0,4225.0,4225.0,36 months,14.85,146.16,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68547291,NaN,24000.0,24000.0,24000.0,60 months,12.88,544.61,C,C2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68547492,NaN,14000.0,14000.0,14000.0,60 months,15.77,338.75,D,D1,...,NaN,NaN,Cash,Y,Jun-2018,ACTIVE,Jun-2018,4486.0,45.0,18.0



Columnas: ['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'acc_n

In [88]:
print("=" * 70)
print("VISTA PREVIA - DATOS RECHAZADOS")
print("=" * 70)
display(df_rej.head(3))
print(f"\nColumnas: {list(df_rej.columns)}")

VISTA PREVIA - DATOS RECHAZADOS


,Amount Requested,Application Date,Loan Title,Risk_Score,Debt-To-Income Ratio,Zip Code,State,Employment Length,Policy Code
0,2000.0,2007-09-20,Honey,665.0,35.4%,850xx,AZ,< 1 year,0.0
1,10000.0,2007-11-19,littlebit0118,561.0,22.33%,275xx,NC,10+ years,0.0
2,6000.0,2007-11-23,sjsmom,496.0,5.6%,746xx,OK,1 year,0.0



Columnas: ['Amount Requested', 'Application Date', 'Loan Title', 'Risk_Score', 'Debt-To-Income Ratio', 'Zip Code', 'State', 'Employment Length', 'Policy Code']


---
## 2. Limpieza de Nombres de Columnas

Los nombres de columna originales mezclan mayusculas, espacios y caracteres
especiales. Estandarizamos a formato `snake_case` para consistencia.

**Transformacion:** `'Loan Amount'` → `'loan_amnt'`, `'Issue Date'` → `'issue_d'`

In [89]:
def clean_column_names(df):
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_").str.replace("-", "_")
    return df

df_acc = clean_column_names(df_acc)
df_rej = clean_column_names(df_rej)

print("Nombres normalizados a snake_case.")
print(f"Aceptados - primeras 10 columnas: {list(df_acc.columns[:10])}")
print(f"Rechazados - columnas: {list(df_rej.columns)}")

Nombres normalizados a snake_case.
Aceptados - primeras 10 columnas: ['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade']
Rechazados - columnas: ['amount_requested', 'application_date', 'loan_title', 'risk_score', 'debt_to_income_ratio', 'zip_code', 'state', 'employment_length', 'policy_code']


---
## 3. Variable Objetivo: Bad Loan

La columna `loan_status` contiene el resultado del prestamo. Creamos una
variable binaria `bad_loan` donde:

- **0 (Good Loan):** Fully Paid, Current, Does not meet policy - Fully Paid
- **1 (Bad Loan):** Charged Off, Default, Late (31-120 days), Late (16-30 days),
  In Grace Period, Does not meet policy - Charged Off

In [90]:
print("Distribucion original de loan_status:")
print(df_acc["loan_status"].value_counts().to_string())

Distribucion original de loan_status:
loan_status
Fully Paid                                             9502
Current                                                7746
Charged Off                                            2412
Late (31-120 days)                                      201
In Grace Period                                          75
Late (16-30 days)                                        41
Does not meet the credit policy. Status:Fully Paid       18
Does not meet the credit policy. Status:Charged Off       4
Default                                                   1


In [91]:
TARGET_MAP = {
    "Fully Paid": 0,
    "Current": 0,
    "Charged Off": 1,
    "Default": 1,
    "Late (31-120 days)": 1,
    "Late (16-30 days)": 1,
    "In Grace Period": 1,
    "Does not meet the credit policy. Status:Fully Paid": 0,
    "Does not meet the credit policy. Status:Charged Off": 1,
}

df_acc["bad_loan"] = df_acc["loan_status"].map(TARGET_MAP)
n_before = len(df_acc)
df_acc = df_acc.dropna(subset=["bad_loan"])
df_acc["bad_loan"] = df_acc["bad_loan"].astype(int)

print(f"Filas eliminadas por loan_status no mapeable: {n_before - len(df_acc)}")
print(f"\nDistribucion de bad_loan:")
print(df_acc["bad_loan"].value_counts())
print(f"\nTasa de default: {df_acc['bad_loan'].mean()*100:.2f}%")

Filas eliminadas por loan_status no mapeable: 0

Distribucion de bad_loan:
bad_loan
0    17266
1     2734
Name: count, dtype: int64

Tasa de default: 13.67%


In [92]:
# --- Validacion: target ---
assert "bad_loan" in df_acc.columns, "bad_loan no creada"
assert df_acc["bad_loan"].dtype == int, "bad_loan debe ser int"
assert df_acc["bad_loan"].isna().sum() == 0, "bad_loan no debe tener NaN"
print(f"target OK: {df_acc['bad_loan'].sum():,} defaults ({df_acc['bad_loan'].mean()*100:.1f}%)")


target OK: 2,734 defaults (13.7%)


In [93]:
# --- Grafico: distribucion bad_loan ---
fig = px.pie(values=df_acc["bad_loan"].value_counts().values,
             names=["Good (0)", "Default (1)"],
             title="Distribucion de prestamos: Good vs Default",
             color_discrete_sequence=["#2ecc71", "#e74c3c"],
             hole=0.4)
fig.update_traces(textinfo="label+percent",
                  textposition="outside")
fig.update_layout(template="plotly_white",
                  width=600, height=400)
fig.show()


---
## 4. Parseo de Tipos de Datos

Numerosas columnas vienen como strings cuando deberian ser numeros o fechas.
Aplicamos las siguientes transformaciones:

| Columna | Original | Transformado |
|---------|----------|--------------|
| `issue_d`, `earliest_cr_line`, etc. | `"Dec-2015"` | `datetime` |
| `int_rate` | `"13.99%"` | `float` (0.1399) |
| `loan_amnt` | `"$36,000"` | `float` |
| `term` | `"36 months"` | `float` (36) |
| `emp_length` | `"10+ years"` | `float` (10) |

In [94]:
# --- Fechas ---
date_cols = ["issue_d", "earliest_cr_line", "last_pymnt_d",
            "last_credit_pull_d", "sec_app_earliest_cr_line"]
for col in date_cols:
    if col in df_acc.columns:
        df_acc[col] = pd.to_datetime(df_acc[col], format="%b-%Y", errors="coerce")

print("Fechas parseadas correctamente.")
for col in date_cols:
    if col in df_acc.columns:
        print(f"  {col}: {df_acc[col].dtype} - rango [{df_acc[col].min()} a {df_acc[col].max()}]")

Fechas parseadas correctamente.
  issue_d: datetime64[ns] - rango [2007-08-01 00:00:00 a 2018-12-01 00:00:00]
  earliest_cr_line: datetime64[ns] - rango [1958-01-01 00:00:00 a 2015-10-01 00:00:00]
  last_pymnt_d: datetime64[ns] - rango [2008-06-01 00:00:00 a 2019-03-01 00:00:00]
  last_credit_pull_d: datetime64[ns] - rango [2007-05-01 00:00:00 a 2019-03-01 00:00:00]
  sec_app_earliest_cr_line: datetime64[ns] - rango [1970-10-01 00:00:00 a 2018-01-01 00:00:00]


In [95]:
# --- Porcentajes ---
def parse_percent(col):
    if not pd.api.types.is_numeric_dtype(col):
        return col.str.replace("%", "", regex=False).astype(float)
    return col
df_acc["int_rate"] = parse_percent(df_acc["int_rate"])
df_acc["revol_util"] = parse_percent(df_acc["revol_util"])

print("Porcentajes parseados:")
print(f"  int_rate: media={df_acc['int_rate'].mean():.4f}, std={df_acc['int_rate'].std():.4f}")
print(f"  revol_util: media={df_acc['revol_util'].mean():.4f}, std={df_acc['revol_util'].std():.4f}")

Porcentajes parseados:
  int_rate: media=13.0919, std=4.8265
  revol_util: media=50.3646, std=24.6701


In [96]:
# --- Validacion: porcentajes ---
assert df_acc["int_rate"].dtype == float, "int_rate debe ser float"
assert df_acc["revol_util"].dtype == float, "revol_util debe ser float"
print(f"porcentajes OK: int_rate range=[{df_acc['int_rate'].min():.4f}, {df_acc['int_rate'].max():.4f}]")


porcentajes OK: int_rate range=[5.3100, 30.9900]


In [97]:
# --- Grafico: distribucion int_rate por bad_loan ---
fig = px.histogram(df_acc, x="int_rate", color="bad_loan",
                   barmode="overlay", opacity=0.6,
                   title="Tasa de interes por estado del prestamo",
                   labels={"int_rate": "Tasa de interes", "bad_loan": "Default"},
                   color_discrete_map={0: "#2ecc71", 1: "#e74c3c"},
                   nbins=50)
fig.update_layout(template="plotly_white",
                  width=800, height=450,
                  legend_title="Default")
fig.show()


In [98]:
# --- Montos monetarios ---
def parse_money(col):
    if not pd.api.types.is_numeric_dtype(col):
        return col.replace(r'[\$,]', '', regex=True).astype(float)
    return col

money_cols = ["loan_amnt", "funded_amnt", "funded_amnt_inv",
              "annual_inc", "installment", "total_pymnt",
              "total_rec_prncp", "total_rec_int", "recoveries",
              "collection_recovery_fee", "last_pymnt_amnt",
              "tot_coll_amt", "tot_cur_bal", "total_rev_hi_lim",
              "avg_cur_bal", "bc_open_to_buy", "tot_hi_cred_lim",
              "total_bal_ex_mort", "total_bc_limit",
              "total_il_high_credit_limit", "annual_inc_joint",
              "revol_bal", "out_prncp", "out_prncp_inv",
              "total_pymnt_inv", "total_rec_late_fee",
              "max_bal_bc", "total_bal_il", "hardship_amount",
              "settlement_amount", "revol_bal_joint"]

for col in money_cols:
    if col in df_acc.columns:
        df_acc[col] = parse_money(df_acc[col])

print("Montos parseados a float.")
print(f"Ejemplo - loan_amnt: min={df_acc['loan_amnt'].min()}, max={df_acc['loan_amnt'].max()}")

Montos parseados a float.
Ejemplo - loan_amnt: min=1000.0, max=40000.0


In [99]:
# --- Plazo (term) ---
def parse_term(col):
    if col.dtype == object:
        return col.str.extract(r"(\d+)", expand=False).astype(float)
    return col

df_acc["term"] = parse_term(df_acc["term"])
print(f"Term (meses): valores={df_acc['term'].unique()}, nulos={df_acc['term'].isnull().sum()}")

Term (meses): valores=[36. 60.], nulos=0


In [100]:
# --- Antiguedad laboral ---
df_acc["emp_length"] = df_acc["emp_length"].str.extract(r"(\d+)", expand=False).astype(float)
print(f"emp_length: media={df_acc['emp_length'].mean():.1f}, nulos={df_acc['emp_length'].isnull().sum()}")

emp_length: media=6.0, nulos=1328


In [101]:
# --- Validacion: numericas ---
assert df_acc["term"].dtype == float, "term debe ser float"
assert df_acc["emp_length"].dtype == float, "emp_length debe ser float"
print(f"numericas OK: term={df_acc['term'].unique()}, emp_length NaN={df_acc['emp_length'].isna().sum():,}")


numericas OK: term=[36. 60.], emp_length NaN=1,328


---
## 5. Feature Engineering

Creamos nuevas variables a partir de columnas existentes:

| Feature | Fuente | Descripcion |
|---------|--------|-------------|
| `fico_score` | `fico_range_low` + `fico_range_high` | Punto medio del rango FICO |
| `had_hardship` | `hardship_flag` | Indica si hubo plan de dificultades |
| `debt_settlement` | `debt_settlement_flag` | Indica si hubo acuerdo de deuda |

In [102]:
# --- FICO Score (midpoint) ---
if "fico_range_low" in df_acc.columns and "fico_range_high" in df_acc.columns:
    df_acc["fico_score"] = (df_acc["fico_range_low"] + df_acc["fico_range_high"]) / 2
print(f"fico_score: media={df_acc['fico_score'].mean():.1f}, rango=[{df_acc['fico_score'].min()}, {df_acc['fico_score'].max()}]")

fico_score: media=700.5, rango=[657.0, 847.5]


In [103]:
# --- Hardship Flag ---
if "hardship_flag" in df_acc.columns:
    df_acc["had_hardship"] = (df_acc["hardship_flag"] == "Y").astype(int)
else:
    df_acc["had_hardship"] = 0
print(f"had_hardship: {df_acc['had_hardship'].sum()} prestamos con hardship")

had_hardship: 8 prestamos con hardship


In [104]:
# --- Debt Settlement Flag ---
if "debt_settlement_flag" in df_acc.columns:
    df_acc["debt_settlement"] = (df_acc["debt_settlement_flag"] == "Y").astype(int)
else:
    df_acc["debt_settlement"] = 0
print(f"debt_settlement: {df_acc['debt_settlement'].sum()} prestamos con settlement")

debt_settlement: 311 prestamos con settlement


---
## 6. Manejo de Valores Faltantes

Aplicamos estrategias especificas segun la naturaleza de cada variable:

| Variable | Estrategia | Justificacion |
|----------|-----------|---------------|
| `mths_since_last_delinq` | NaN si 0 (nunca moroso) | 0 indica "ninguna" → NaN evita sesgo |
| `mths_since_last_record` | NaN si 0 (sin incumplimiento grave) | Misma logica |
| `pub_rec_bankruptcies` | fillna(0) → int | NaN asumido como 0 quiebras |
| Columnas co-solicitante | Se conservan con NaN | Esperable: ~90% son individuales |

In [105]:
df_acc["had_delinquency"] = (df_acc["mths_since_last_delinq"].notna() & (df_acc["mths_since_last_delinq"] > 0)).astype(int)
df_acc["mths_since_last_delinq"] = df_acc["mths_since_last_delinq"].replace(0, np.nan)
df_acc["mths_since_last_record"] = df_acc["mths_since_last_record"].replace(0, np.nan)
df_acc["pub_rec_bankruptcies"] = df_acc["pub_rec_bankruptcies"].fillna(0).astype(int)

print("Valores faltantes tratados.")

# Diagnostico post-tratamiento
miss = df_acc.isnull().sum()
miss_pct = (miss / len(df_acc)) * 100
miss_report = pd.DataFrame({"Missing": miss, "%": miss_pct.round(1)})
miss_report = miss_report[miss_report["Missing"] > 0].sort_values("%", ascending=False)
print(f"\nColumnas con missing post-tratamiento: {len(miss_report)}")
print("\nTop 15:")
print(miss_report.head(15).to_string())

Valores faltantes tratados.

Columnas con missing post-tratamiento: 110

Top 15:
                                            Missing      %
member_id                                     20000  100.0
orig_projected_additional_accrued_interest    19924   99.6
hardship_start_date                           19906   99.5
hardship_end_date                             19906   99.5
hardship_status                               19906   99.5
hardship_type                                 19906   99.5
hardship_amount                               19906   99.5
hardship_dpd                                  19906   99.5
hardship_length                               19906   99.5
payment_plan_start_date                       19906   99.5
hardship_last_payment_amount                  19906   99.5
hardship_payoff_balance_amount                19906   99.5
deferral_term                                 19906   99.5
hardship_reason                               19906   99.5
hardship_loan_status              

In [106]:
# --- Validacion: missing values ---
assert "had_delinquency" in df_acc.columns, "had_delinquency no creada"
assert df_acc["had_delinquency"].dtype == int, "had_delinquency debe ser int"
print(f"missing OK: had_delinquency media={df_acc['had_delinquency'].mean():.3f}")


missing OK: had_delinquency media=0.489


In [107]:
# --- Grafico: missing values post-tratamiento ---
miss = df_acc.isnull().sum()
miss_pct = (miss / len(df_acc)) * 100
miss_df = pd.DataFrame({"col": miss.index, "pct": miss_pct.values})
miss_df = miss_df[miss_df["pct"] > 0].sort_values("pct", ascending=False).head(20)

fig = px.bar(miss_df, x="pct", y="col", orientation="h",
             title="Top 20 columnas con valores faltantes post-tratamiento",
             labels={"pct": "% Missing", "col": "Columna"},
             color="pct", color_continuous_scale="Reds_r")
fig.update_layout(template="plotly_white",
                  width=800, height=500,
                  yaxis={"categoryorder": "total ascending"})
fig.show()


---
## 7. Eliminacion de Columnas con Leakage

El **leakage** ocurre cuando variables disponibles solo despues del evento
 (default/pago) se usan para predecirlo. Esto infla artificialmente las metricas.

Eliminamos:

1. **Identificadores:** `id`, `member_id`, `url`, `desc`, `title`, `zip_code`, `emp_title`
2. **Post-prestamo:** `out_prncp*`, `total_pymnt*`, `total_rec*`, `last_pymnt*`,
   `last_fico*`, `recoveries`, `collection_*`
3. **Hardship/Settlement post-evento:** `hardship_*`, `settlement_*`, `debt_settlement_*`
4. **Fechas futuras:** `next_pymnt_d`, `hardship_start_date`, `settlement_date`

In [108]:
DROP_COLS = [
    "id", "member_id", "url", "desc", "title",
    "zip_code", "emp_title",
    "hardship_type", "hardship_reason", "hardship_status",
    "hardship_start_date", "hardship_end_date",
    "payment_plan_start_date",
    "debt_settlement_flag_date", "settlement_status",
    "settlement_date",
    "orig_projected_additional_accrued_interest",
    "hardship_payoff_balance_amount", "hardship_last_payment_amount",
    "next_pymnt_d",
    "hardship_loan_status",
]

post_loan_prefixes = [
    "out_prncp", "total_pymnt", "total_rec",
    "last_pymnt", "last_fico", "recoveries",
    "collection_", "next_", "hardship_",
    "settlement_", "debt_settlement_",
]

cols_before = df_acc.shape[1]

explicit_drop = [c for c in DROP_COLS if c in df_acc.columns]
prefix_drop = [c for c in df_acc.columns if any(c.startswith(p) for p in post_loan_prefixes)]
all_drop = list(set(explicit_drop + prefix_drop))

df_acc = df_acc.drop(columns=all_drop, errors="ignore")

print(f"Columnas eliminadas por leakage: {len(all_drop)}")
print(f"Columnas restantes: {cols_before} -> {df_acc.shape[1]}")

Columnas eliminadas por leakage: 42
Columnas restantes: 156 -> 114


In [109]:
# --- Validacion: leakage ---
leakage_cols = [c for c in df_acc.columns if c.startswith(("total_pymnt", "total_rec", "last_pymnt", "out_prncp", "recoveries", "settlement_"))]
assert len(leakage_cols) == 0, f"Leakage aun presente: {leakage_cols}"
print(f"leakage OK: {df_acc.shape[1]} columnas restantes")


leakage OK: 114 columnas restantes


In [110]:
print("Columnas eliminadas:")
for c in sorted(all_drop):
    print(f"  - {c}")

Columnas eliminadas:
  - collection_recovery_fee
  - debt_settlement_flag
  - debt_settlement_flag_date
  - desc
  - emp_title
  - hardship_amount
  - hardship_dpd
  - hardship_end_date
  - hardship_flag
  - hardship_last_payment_amount
  - hardship_length
  - hardship_loan_status
  - hardship_payoff_balance_amount
  - hardship_reason
  - hardship_start_date
  - hardship_status
  - hardship_type
  - id
  - last_fico_range_high
  - last_fico_range_low
  - last_pymnt_amnt
  - last_pymnt_d
  - member_id
  - next_pymnt_d
  - orig_projected_additional_accrued_interest
  - out_prncp
  - out_prncp_inv
  - payment_plan_start_date
  - recoveries
  - settlement_amount
  - settlement_date
  - settlement_percentage
  - settlement_status
  - settlement_term
  - title
  - total_pymnt
  - total_pymnt_inv
  - total_rec_int
  - total_rec_late_fee
  - total_rec_prncp
  - url
  - zip_code



---
## 8. Integracion de Datos Macroeconomicos (FRED API)

Cargamos indicadores macroeconomicos descargados de FRED y los mergeamos por (year, month) de `issue_d`.
Estas variables enriquecen el dataset para mejorar el modelo predictivo.

In [111]:
# --- Cargar datos macroeconomicos FRED ---
EXTERNAL_DIR = "../data/external"
FRED_PATH = os.path.join(EXTERNAL_DIR, "fred_indicators.csv")
if os.path.exists(FRED_PATH):
    fred = pd.read_csv(FRED_PATH)
    fred["year"] = fred["year"].astype(int)
    fred["month"] = fred["month"].astype(int)
    df_acc["year"] = df_acc["issue_d"].dt.year
    df_acc["month"] = df_acc["issue_d"].dt.month
    df_acc = df_acc.merge(fred[["year","month","unrate","fed_funds","cpi"]],
                          on=["year","month"], how="left")
    print(f"FRED indicators merged: {df_acc.shape}")
    print(df_acc[["unrate","fed_funds","cpi"]].describe())
else:
    print("FRED data not found. Run scripts/fetch_fred.py first.")


FRED indicators merged: (20000, 119)
             unrate     fed_funds           cpi
count  20000.000000  20000.000000  20000.000000
mean       5.050280      0.727534    241.564362
std        1.168971      0.698830      6.815792
min        3.700000      0.070000    207.667000
25%        4.200000      0.120000    237.430000
50%        4.900000      0.390000    240.222000
75%        5.400000      1.160000    247.284000
max       10.000000      5.020000    252.772000


---
## 9. Preprocesamiento de Datos Rechazados

Los datos rechazados tienen solo 9 columnas y no incluyen outcome.
Sirven principalmente para analisis descriptivo comparativo.

Transformaciones:
- Parseo de fechas (`application_date`)
- Parseo de montos (`amount_requested`)
- Parseo de DTI y Risk Score
- Flag `rejected = 1`

In [112]:
print("Columnas originales rechazados:")
for col in df_rej.columns:
    print(f"  {col}: dtype={df_rej[col].dtype}")

Columnas originales rechazados:
  amount_requested: dtype=float64
  application_date: dtype=object
  loan_title: dtype=object
  risk_score: dtype=float64
  debt_to_income_ratio: dtype=object
  zip_code: dtype=object
  state: dtype=object
  employment_length: dtype=object
  policy_code: dtype=float64


In [113]:
# Parseo de fechas
date_cols_rej = [c for c in df_rej.columns if "date" in c]
for col in date_cols_rej:
    if col in df_rej.columns:
        df_rej[col] = pd.to_datetime(df_rej[col], format="%m/%d/%Y", errors="coerce")

# Parseo de montos
if "amount_requested" in df_rej.columns:
    df_rej["amount_requested"] = parse_money(df_rej["amount_requested"])

# Parseo de DTI
target_dti_col = [c for c in df_rej.columns if "debt" in c and "income" in c]
if target_dti_col:
    source_col = target_dti_col[0]
    df_rej["dti"] = df_rej[source_col].str.replace("%", "", regex=False).astype(float)
    df_rej["dti"] = df_rej["dti"].clip(upper=200.0)
    df_rej = df_rej.drop(columns=[source_col])

# Parseo de Risk Score
if "risk_score" in df_rej.columns:
    df_rej["risk_score"] = pd.to_numeric(df_rej["risk_score"], errors="coerce")

df_rej["rejected"] = 1

print(f"Shape post-pipeline rechazados: {df_rej.shape}")
print(f"\nTipos de datos finales:")
print(df_rej.dtypes.to_string())

Shape post-pipeline rechazados: (20000, 10)

Tipos de datos finales:
amount_requested            float64
application_date     datetime64[ns]
loan_title                   object
risk_score                  float64
zip_code                     object
state                        object
employment_length            object
policy_code                 float64
dti                         float64
rejected                      int64


---
## 10. Exportacion a Processed

Guardamos los datasets limpios en `data/processed/`:

| Archivo | Contenido | Filas | Columnas |
|---------|-----------|-------|----------|
| `accepted_clean.csv` | Prestamos aceptados limpios | 5,000 | ~113 |
| `rejected_clean.csv` | Solicitudes rechazadas | 5,000 | ~9 |
| `combined_summary.csv` | Union para analisis comparativo | 10,000 | ~10 |

In [114]:
df_acc.to_csv(os.path.join(PROCESSED_DIR, "accepted_clean.csv"), index=False)
df_rej.to_csv(os.path.join(PROCESSED_DIR, "rejected_clean.csv"), index=False)

print(f"accepted_clean.csv -> {df_acc.shape}")
print(f"rejected_clean.csv -> {df_rej.shape}")

accepted_clean.csv -> (20000, 119)
rejected_clean.csv -> (20000, 10)


---
## 11. Dataset Combinado (Aceptados + Rechazados)

Creamos un dataset unificado con las columnas comunes entre aceptados y
rechazados para analisis comparativo y potencial uso en modelos de
aprobacion/ rechazo.

In [115]:
# Construir subset de rechazados con columnas homologas
rej_cols = {}
if "amount_requested" in df_rej.columns: rej_cols["loan_amnt"] = df_rej["amount_requested"]
if "dti" in df_rej.columns: rej_cols["dti"] = df_rej["dti"]
if "risk_score" in df_rej.columns: rej_cols["fico_score"] = df_rej["risk_score"]
if "state" in df_rej.columns: rej_cols["addr_state"] = df_rej["state"]
if "employment_length" in df_rej.columns: rej_cols["emp_length"] = df_rej["employment_length"]
if "application_date" in df_rej.columns: rej_cols["issue_d"] = df_rej["application_date"]
rej_cols["rejected"] = 1
df_rej_summary = pd.DataFrame(rej_cols)

# Subset de aceptados
common_cols = ["loan_amnt", "dti", "fico_score", "addr_state",
               "emp_length", "issue_d", "bad_loan"]
extra_cols = ["int_rate", "grade", "annual_inc", "home_ownership"]
available_extra = [c for c in extra_cols if c in df_acc.columns]

df_acc_summary = df_acc[common_cols + available_extra].copy()
df_acc_summary["rejected"] = 0

combined = pd.concat([df_acc_summary, df_rej_summary], ignore_index=True, sort=False)

combined.to_csv(os.path.join(PROCESSED_DIR, "combined_summary.csv"), index=False)

print(f"combined_summary.csv -> {combined.shape}")
print(f"\nDistribucion aceptados vs rechazados:")
print(combined["rejected"].value_counts().to_string())

combined_summary.csv -> (40000, 12)

Distribucion aceptados vs rechazados:
rejected
0    20000
1    20000


In [116]:
print("=" * 70)
print("RESUMEN FINAL DEL PIPELINE")
print("=" * 70)
print(f"\nDataset ACEPTADOS:")
print(f"  Shape: {df_acc.shape}")
print(f"  Columnas: {df_acc.shape[1]}")
print(f"  Tipos: {df_acc.dtypes.value_counts().to_dict()}")
print(f"  Missing total: {df_acc.isnull().sum().sum():,} celdas")
print(f"  Target bad_loan: {df_acc['bad_loan'].sum()} bajos ({df_acc['bad_loan'].mean()*100:.1f}%)")
print(f"\nDataset RECHAZADOS:")
print(f"  Shape: {df_rej.shape}")
print(f"  Columnas: {df_rej.shape[1]}")
print(f"\nDataset COMBINADO:")
print(f"  Shape: {combined.shape}")
print(f"\nArchivos generados en {PROCESSED_DIR}/:")
for f in os.listdir(PROCESSED_DIR):
    fpath = os.path.join(PROCESSED_DIR, f)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f"  {f} ({size_mb:.2f} MB)")

RESUMEN FINAL DEL PIPELINE

Dataset ACEPTADOS:
  Shape: (20000, 119)
  Columnas: 119
  Tipos: {dtype('float64'): 96, dtype('O'): 12, dtype('int64'): 5, dtype('<M8[ns]'): 4, dtype('int32'): 2}
  Missing total: 530,225 celdas
  Target bad_loan: 2734 bajos (13.7%)

Dataset RECHAZADOS:
  Shape: (20000, 10)
  Columnas: 10

Dataset COMBINADO:
  Shape: (40000, 12)

Archivos generados en ../data/processed/:
  accepted_clean.csv (10.05 MB)
  combined_summary.csv (1.98 MB)
  rejected_clean.csv (1.08 MB)


In [117]:
# --- Dashboard final: métricas clave (versión mejorada) ---

# Paleta coherente y con buen contraste
COLOR_GOOD    = "#2ecc71"
COLOR_BAD     = "#e74c3c"
COLOR_LOAN    = "#5B8FF9"
COLOR_FICO    = "#8E7CC3"
COLOR_TERM    = ["#1abc9c", "#f39c12"]
BG_COLOR      = "#FAFAFA"
FONT_COLOR    = "#2c3e50"
FONT_FAMILY   = "Inter, Helvetica, Arial, sans-serif"

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Tasa de default", "Distribución del monto del préstamo",
                     "Distribución del puntaje FICO", "Distribución del plazo"),
    specs=[[{"type": "pie"}, {"type": "histogram"}],
           [{"type": "histogram"}, {"type": "pie"}]],
    horizontal_spacing=0.12,
    vertical_spacing=0.18,
)

# 1. Default rate
default_counts = df_acc["bad_loan"].value_counts()
default_rate = default_counts.get(1, 0) / default_counts.sum() * 100

fig.add_trace(go.Pie(
    labels=["Bueno", "Default"],
    values=default_counts.values,
    marker=dict(colors=[COLOR_GOOD, COLOR_BAD], line=dict(color="white", width=2)),
    textinfo="percent",
    textfont=dict(size=13, color="white"),
    hole=0.55,
    showlegend=True,
    legendgroup="default",
    domain={"x": [0.0, 0.45], "y": [0.55, 1.0]},
), row=1, col=1)

# Anotación central con la tasa de default
fig.add_annotation(
    text=f"<b>{default_rate:.1f}%</b><br><span style='font-size:11px'>default</span>",
    x=0.225, y=0.775, xref="paper", yref="paper",
    showarrow=False, font=dict(size=18, color=FONT_COLOR, family=FONT_FAMILY),
)

# 2. Loan amount
fig.add_trace(go.Histogram(
    x=df_acc["loan_amnt"],
    marker=dict(color=COLOR_LOAN, line=dict(color="white", width=0.3)),
    nbinsx=50, name="Monto", opacity=0.9,
), row=1, col=2)

# 3. FICO score
fig.add_trace(go.Histogram(
    x=df_acc["fico_score"],
    marker=dict(color=COLOR_FICO, line=dict(color="white", width=0.3)),
    nbinsx=50, name="FICO", opacity=0.9,
), row=2, col=1)

# 4. Term
term_counts = df_acc["term"].value_counts().sort_index()
fig.add_trace(go.Pie(
    labels=[f"{int(t)} meses" for t in term_counts.index],
    values=term_counts.values,
    marker=dict(colors=COLOR_TERM, line=dict(color="white", width=2)),
    textinfo="percent",
    textfont=dict(size=13, color="white"),
    hole=0.55,
    showlegend=True,
    legendgroup="term",
    domain={"x": [0.55, 1.0], "y": [0.0, 0.45]},
), row=2, col=2)

# Layout general
fig.update_layout(
    template="plotly_white",
    width=950, height=650,
    title=dict(
        text="<b>Dashboard: Resumen de préstamos aceptados</b>",
        x=0.5, xanchor="center",
        font=dict(size=22, color=FONT_COLOR, family=FONT_FAMILY),
    ),
    font=dict(family=FONT_FAMILY, color=FONT_COLOR, size=12),
    plot_bgcolor=BG_COLOR,
    paper_bgcolor="white",
    bargap=0.05,
    legend=dict(orientation="h", yanchor="bottom", y=-0.08, x=0.5, xanchor="center"),
    margin=dict(t=100, l=60, r=60, b=60),
)

# Ejes con estilo consistente
fig.update_xaxes(showgrid=True, gridcolor="#EDEDED", zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor="#EDEDED", zeroline=False)
fig.update_xaxes(title_text="Monto ($)", row=1, col=2)
fig.update_xaxes(title_text="Puntaje FICO", row=2, col=1)
fig.update_yaxes(title_text="Frecuencia", row=1, col=2)
fig.update_yaxes(title_text="Frecuencia", row=2, col=1)

# Estilo de los títulos de cada subplot
fig.update_annotations(font=dict(size=14, color=FONT_COLOR, family=FONT_FAMILY))

fig.show()

---
## Proximo Paso: Modelado

Los datos procesados estan listos para la etapa de modelado en
`notebooks/03_modeling.ipynb`. Los pasos siguientes incluyen:

1. **Separacion train/test** estratificada por `bad_loan`
2. **Imputacion** de valores faltantes (media/mediana/moda segun variable)
3. **Escalado** de variables numericas (StandardScaler / MinMaxScaler)
4. **Codificacion** de variables categoricas (OneHot / Target encoding)
5. **Balanceo** con SMOTE para manejar desbalance de clases
6. **Entrenamiento** de modelos baseline (Logistic Regression, Random Forest, XGBoost)
7. **Evaluacion** con metricas: AUC-ROC, Precision, Recall, F1

---

### Resumen de Transformaciones Aplicadas

| # | Etapa | Descripcion | Impacto |
|---|-------|-------------|---------|
| 1 | Columnas | Normalizacion a snake_case | Consistencia |
| 2 | Target | Mapeo loan_status -> bad_loan binario | ~XX% malos / ~XX% buenos |
| 3 | Fechas | str -> datetime (5 columnas) | 0 perdidas por coerce |
| 4 | Porcentajes | str -> float (int_rate, revol_util) | 0 perdidas |
| 5 | Montos | str -> float (31 columnas) | 0 perdidas |
| 6 | Term/Emp | str -> float (extract digits) | 0 perdidas |
| 7 | Features | fico_score, had_hardship, debt_settlement | +3 columnas |
| 8 | Missing | replace(0, NaN) en delinq/record | ~XX% missing en cada una |
| 9 | Leakage | Drop de ~42 columnas post-evento | 151 -> ~113 columnas |

---

*Notebook generado automaticamente para el proyecto integrador de*
*Programacion para la Ciencia de Datos - Julio 2026*